# Loss-based difficulty: analysis

Looks at `data/features/alma_train_loss.csv` (from `scripts/compute_loss.py`): the per-token
loss of every ALMA training pair under the **pretrained** M2M-100.

Questions:
1. What does the `loss_mean` distribution look like (where would a "hard" cutoff go)?
2. How much does it overlap with the difficulty we already have (length, `rarity_mean`)?
3. What are the highest-loss pairs: genuinely hard, or noise (misaligned / bad references)?
4. What would a cutoff do to the sampling weights?

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paths are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "scripts")
from data_utils import MAX_LEN, all_cells, assign_bins, read_jsonl

STEM = "alma_train"
LOSS_PATH = Path(f"data/features/{STEM}_loss.csv")

loss = pd.read_csv(LOSS_PATH)
feats = assign_bins(pd.read_csv(f"data/features/{STEM}.csv"))
text = pd.DataFrame(read_jsonl(f"data/processed/{STEM}.jsonl"))[["id", "zh", "en"]]

df = (loss.merge(feats, on="id", validate="one_to_one")
          .merge(text, on="id", validate="one_to_one"))
df["len_ratio"] = df["tgt_len"] / df["src_len"]   # target / source subwords
print(f"{len(loss)} loss rows, {len(feats)} feature rows, {len(df)} merged")
assert len(df) == len(loss) == len(feats), "ids do not match between files"

## 1. Distribution

In [ ]:
pcts = [0.01, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99]
display(df[["loss_mean", "loss_sum", "n_tgt"]].describe(percentiles=pcts).round(3))
print(f"pairs with target truncated at MAX_LEN ({MAX_LEN}): {(df['n_tgt'] >= MAX_LEN).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["loss_mean"], bins=100)
for q in [0.8, 0.9, 0.95]:
    v = df["loss_mean"].quantile(q)
    axes[0].axvline(v, ls="--", c="k", lw=0.8)
    axes[0].text(v, axes[0].get_ylim()[1] * 0.9, f" p{int(q*100)}", fontsize=8)
axes[0].set(xlabel="loss_mean (nats / target token)", ylabel="pairs", title="Per-token loss")
axes[1].hist(df["loss_sum"], bins=100)
axes[1].set(xlabel="loss_sum (nats)", title="Summed loss (grows with length)")
plt.tight_layout()

## 2. Overlap with the existing difficulty features

Spearman (rank) correlation is the main number: we only care whether the *ordering* of pairs
agrees, and it is robust to the long tail. If `loss_mean` vs `rarity_mean` is high (say > 0.7),
the loss condition mostly repeats the rarity condition.

In [ ]:
cols = ["loss_mean", "loss_sum", "rarity_mean", "rarity_max", "src_len", "tgt_len", "len_ratio", "n_unseen"]
spearman = df[cols].corr(method="spearman")
display(spearman.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
print("Pearson loss_mean vs rarity_mean:", round(df["loss_mean"].corr(df["rarity_mean"]), 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, x in zip(axes, ["rarity_mean", "src_len", "len_ratio"]):
    hb = ax.hexbin(df[x], df["loss_mean"], gridsize=60, bins="log", mincnt=1,
                   xscale="log" if x != "rarity_mean" else "linear")
    rho = df[x].rank().corr(df["loss_mean"].rank())  # Spearman without scipy
    ax.set(xlabel=x, ylabel="loss_mean", title=f"loss_mean vs {x} (Spearman {rho:.2f})")
plt.colorbar(hb, ax=axes[-1], label="pairs (log)")
plt.tight_layout()

Mean `loss_mean` per difficulty cell (rows = length bin, columns = rarity bin), and pair counts.

In [ ]:
by_cell = df.groupby(["length_bin", "rarity_bin"])["loss_mean"].agg(["mean", "median", "count"])
order = [c.split("/")[0] for c in all_cells()[::2]]
display(by_cell["mean"].unstack().loc[order].round(3))
display(by_cell["count"].unstack().loc[order])

Do the "hard by loss" pairs and the "rare" pairs pick out the same sentences?
Here "hard" = top 10% by `loss_mean`, so both groups are about 10% of the pool.

In [ ]:
HARD_Q = 0.90
cut = df["loss_mean"].quantile(HARD_Q)
df["hard"] = df["loss_mean"] > cut
display(pd.crosstab(df["hard"], df["rarity_bin"], margins=True))
both = (df["hard"] & (df["rarity_bin"] == "rare")).sum()
either = (df["hard"] | (df["rarity_bin"] == "rare")).sum()
print(f"cutoff p{int(HARD_Q*100)} = {cut:.3f}")
print(f"Jaccard(hard, rare) = {both / either:.2f}  (random overlap would be about 0.05)")
print(f"P(rare | hard) = {both / df['hard'].sum():.2f}   P(rare) = {(df['rarity_bin'] == 'rare').mean():.2f}")

## 3. Highest-loss pairs

Read these. If most look misaligned, partially translated or wrong, upsampling the top would mostly
upsample noise, and we should drop the extreme tail (e.g. top 1%) or put the cutoff band lower.

In [ ]:
pd.set_option("display.max_colwidth", 200)
show = ["id", "loss_mean", "n_tgt", "src_len", "tgt_len", "rarity_mean", "zh", "en"]
display(df.nlargest(30, "loss_mean")[show].round(2))

Lowest-loss pairs, for contrast (often very short, formulaic, or memorised).

In [ ]:
display(df.nsmallest(15, "loss_mean")[show].round(2))

A random sample from the band that would actually get upsampled with a p90 cutoff but no
noise filter (p90 to p99). This is a fairer picture than the extreme top.

In [ ]:
lo, hi = df["loss_mean"].quantile([0.90, 0.99])
band = df[(df["loss_mean"] > lo) & (df["loss_mean"] <= hi)]
display(band.sample(20, random_state=0)[show].round(2))

### Simple noise flags by loss band

Rough automatic checks (none of them is a proof of noise):
- `cjk_in_en`: the English side still contains Chinese characters
- `odd_ratio`: target/source subword ratio outside [0.5, 2.5]
- `very_short`: source of at most 3 subwords

In [ ]:
df["cjk_in_en"] = df["en"].str.contains("[一-鿿]")  # CJK unified ideographs
df["odd_ratio"] = ~df["len_ratio"].between(0.5, 2.5)
df["very_short"] = df["src_len"] <= 3
bands = pd.cut(df["loss_mean"].rank(pct=True), [0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0],
               labels=["0-50%", "50-80%", "80-90%", "90-95%", "95-99%", "top 1%"])
flags = ["cjk_in_en", "odd_ratio", "very_short"]
display((df.groupby(bands, observed=True)[flags].mean() * 100).round(1).add_suffix(" %"))

## 4. What a cutoff would do to sampling

`train.py` would split pairs into easy / hard and use `balanced_weights(..., cap)`: each group gets
half the probability, but no pair more than `cap` (3) times its uniform weight. Below: for a few
cutoffs, the share of hard pairs in the pool and in the drawn batches, on the size-1000 subsets.

In [ ]:
def balanced_weights(groups, cap=3.0):
    # Copy of train.py's balanced_weights (train.py imports torch).
    n_groups = groups.nunique()
    group_size = groups.map(groups.value_counts())
    return ((len(groups) / n_groups) / group_size).clip(upper=cap)

rows = []
for q in [0.80, 0.90, 0.95]:
    cut = df["loss_mean"].quantile(q)       # fixed edge from the FULL pool, like rarity's 7.9
    for size in ["1000", "5000", "full"]:
        for seed in [0, 1, 2]:
            if size == "full":
                sub = df
            else:
                ids = Path(f"data/subsets/seed{seed}_n{size}.txt").read_text().split()
                sub = df.set_index("id").loc[ids]
            hard = sub["loss_mean"] > cut
            w = balanced_weights(hard)
            rows.append({"cutoff": f"p{int(q*100)} ({cut:.2f})", "size": size, "seed": seed,
                         "n_hard": int(hard.sum()), "hard % in pool": 100 * hard.mean(),
                         "hard % in batches": 100 * w[hard].sum() / w.sum(),
                         "max weight": w.max()})
            if size == "full":
                break
display(pd.DataFrame(rows).groupby(["cutoff", "size"], sort=False)
        [["n_hard", "hard % in pool", "hard % in batches", "max weight"]].mean().round(1))

## Notes for the decision

- Correlation with `rarity_mean` / length: _fill in_
- What the top tail looks like (noise or hard): _fill in_
- Chosen cutoff and whether to drop the top 1%: _fill in_